### Base

In [ ]:
import os
import random
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from abc import ABC, abstractmethod
import hmac
import hashlib

### Classes


In [ ]:
class Cipher_(ABC):
    @abstractmethod
    def keygen(self) -> bytes:
        pass
  
    @abstractmethod
    def enc(self, key: bytes, text: bytes) -> bytes:
        pass

    @abstractmethod
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass

class INDCPA_Adv(ABC):
    @abstractmethod
    def choose(self, oracle: callable) -> bytes:
        pass

    @abstractmethod
    def guess(self, oracle: callable, ciphertext: bytes) -> bool:
        pass
    
# CIFRA IDENTIDADE (Insegura)

class IdentityCipher(Cipher_):
    def keygen(self) -> bytes:
        return b''
    
    def enc(self, key: bytes, text: bytes) -> bytes:
        return text
    
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        return ciphertext

class ObviousAdv(INDCPA_Adv):
    def __init__(self):
        self.m0 = b"MENSAGEM_ZERO_"
        self.m1 = b"MENSAGEM_UM___"

    def choose(self, oracle: callable) -> tuple[bytes, bytes]:
        return self.m0, self.m1

    def guess(self, oracle: callable, ciphertext: bytes) -> bool:
        # Como a cifra não altera a mensagem, basta verificar se o texto cifrado recebido é igual à mensagem m1.
        return ciphertext == self.m1
    
# CIFRA ChaCha20 (Segura)
    
class ChaCha20Cipher(Cipher_):
    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, key: bytes, text: bytes) -> bytes:
        nonce = os.urandom(16)
        cipher = Cipher(algorithms.ChaCha20(key, nonce), mode=None)
        encryptor = cipher.encryptor()
        ct = encryptor.update(text) + encryptor.finalize()
        return nonce + ct

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        nonce = ciphertext[:16]
        ct = ciphertext[16:]
        cipher = Cipher(algorithms.ChaCha20(key, nonce), mode=None)
        decryptor = cipher.decryptor()
        return decryptor.update(ct) + decryptor.finalize()

class RandomAdv(INDCPA_Adv):
    def choose(self, oracle: callable) -> tuple[bytes, bytes]:
        return b"MENSAGEM_ZERO_", b"MENSAGEM_UM___"

    def guess(self, oracle: callable, ciphertext: bytes) -> bool:
        # o adversário não aprende nada com o criptograma, logo atira uma moeda ao ar.
        return random.choice([True, False])
    
    

### Prog: indcpa

In [ ]:
def IND_CPA(C: Cipher_, A: INDCPA_Adv) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    
    m = A.choose(enc_oracle)
    assert len(m[0]) == len(m[1]), "As mensagens têm de ter o mesmo tamanho!"
    
    b = random.choice([0, 1])
    
    c = C.enc(k, m[b])
    
    b_prime = A.guess(enc_oracle, c)
    
    return b == int(b_prime)

def calcular_vantagem(C: Cipher_, A: INDCPA_Adv, iteracoes: int = 10000):
    """
    Executa o jogo várias vezes para calcular a probabilidade e a vantagem.
    """
    vitorias = sum(1 for _ in range(iteracoes) if IND_CPA(C, A))
    probabilidade = vitorias / iteracoes
    # Vantagem = 2 * | Pr[ganhar] - 1/2 |
    vantagem = 2 * abs(probabilidade - 0.5)
    
    print(f"Iterações: {iteracoes}")
    print(f"Vitórias: {vitorias} ({probabilidade*100:.2f}%)")
    print(f"Vantagem do Adversário: {vantagem:.4f}\n")
    
if __name__ == '__main__':
    print("--- Cifra Identidade vs Estratégia Óbvia ---")
    cifra_id = IdentityCipher()
    adv_obvio = ObviousAdv()
    calcular_vantagem(cifra_id, adv_obvio, iteracoes=1000)

    print("--- Cifra ChaCha20 vs Estratégia Aleatória ---")
    cifra_chacha = ChaCha20Cipher()
    adv_aleatorio = RandomAdv()
    # ganha ~50% das vezes
    calcular_vantagem(cifra_chacha, adv_aleatorio, iteracoes=10000)

### Prog: indcpa_attack

In [ ]:
class InsecureChaCha20(Cipher_):
    def __init__(self):
        # O erro está aqui: o Nonce é fixo para toda a instância
        self.fixed_nonce = b'\x00' * 16 

    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, key: bytes, text: bytes) -> bytes:
        algorithm = algorithms.ChaCha20(key, self.fixed_nonce)
        cipher = Cipher(algorithm, mode=None)
        encryptor = cipher.encryptor()
        return encryptor.update(text)
    
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        algorithm = algorithms.ChaCha20(key, self.fixed_nonce)
        cipher = Cipher(algorithm, mode=None)
        decryptor = cipher.decryptor()
        return decryptor.update(ciphertext)

class Attacker(INDCPA_Adv):
    def __init__(self):
        self.m0 = b"Mensagem Secreta A"
        self.m1 = b"Mensagem Secreta B"

    def choose(self, oracle):
        return self.m0, self.m1

    def guess(self, oracle, challenge_ct):
        # Como o nonce é fixo, o adversário pede ao oráculo para cifrar m0. Se o resultado for igual ao desafio, b era 0.
        c0 = oracle(self.m0)
        
        if c0 == challenge_ct:
            return 0
        else:
            return 1

In [ ]:
# Simulador do Jogo

def run_ind_cpa_game(C: Cipher_, A: INDCPA_Adv):
    import random
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1)
    
    b = random.randint(0, 1)
    m_chosen = m0 if b == 0 else m1
    
    c = C.enc(k, m_chosen)
    
    b_prime = A.guess(enc_oracle, c)
    
    return b == b_prime

if __name__ == "__main__":
    cifra_insegura = InsecureChaCha20()
    atacante = Attacker()
    
    vitorias = 0
    testes = 100
    for _ in range(testes):
        if run_ind_cpa_game(cifra_insegura, atacante):
            vitorias += 1
            
    print(f"Resultado do Ataque: {vitorias}/{testes} vitórias.")
    print("A vantagem do adversário é de 100% devido ao Nonce fixo!")

No problema a cifra ChaCha20 gera uma sequência de números aleatórios baseada na key e no nonce. Se o nonce nunca muda, a sequência gerada para XOR é sempre a mesma.

No ataque o método guess, o atacante simplesmente usa o Oráculo para cifrar uma das suas mensagens originais (m0). Como a chave e o nonce no sistema não mudaram, o resultado (c0) será exatamente igual ao desafio se o bit escolhido pelo jogo tiver sido o 0.

In [ ]:
# INSEGURANÇA DO ENCRYPT & MAC

class EncryptAndMAC(Cipher_):
    def keygen(self) -> bytes:
        # Precisamos de duas chaves uma para cifrar (32 bytes) e outra para o MAC (32 bytes)
        return os.urandom(64)

    def enc(self, key: bytes, text: bytes) -> bytes:
        key_enc = key[:32]
        key_mac = key[32:]

        # Usamos o ChaCha20 (seguro) como no exemplo anterior
        nonce = os.urandom(16)
        cipher = Cipher(algorithms.ChaCha20(key_enc, nonce), mode=None)
        encryptor = cipher.encryptor()
        C = nonce + encryptor.update(text) + encryptor.finalize()

        # Calculamos o MAC sobre o TEXTO-LIMPO
        # Usamos HMAC com SHA256, que produz uma tag sempre com 32 bytes
        T = hmac.new(key_mac, text, hashlib.sha256).digest()

        return C + T

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass


class EncryptAndMAC_Adv(INDCPA_Adv):
    def __init__(self): 
        self.m0 = b"MENSAGEM_SECRETA_0"
        self.m1 = b"MENSAGEM_SECRETA_1"
        self.mac_m0_guardado = b""

    def choose(self, oracle: callable) -> tuple[bytes, bytes]:
        criptograma_oraculo = oracle(self.m0)
        # Sabemos que o MAC tem 32 bytes e é colado no fim
        self.mac_m0_guardado = criptograma_oraculo[-32:]
        
        return self.m0, self.m1

    def guess(self, oracle: callable, ciphertext: bytes) -> bool:
        # Extraímos os últimos 32 bytes
        mac_desafio = ciphertext[-32:]
        
        if mac_desafio == self.mac_m0_guardado:
            return False
        else:
            return True

if __name__ == '__main__':

    print("--- Encrypt & MAC vs Adversário Inteligente ---")
    cifra_enc_mac = EncryptAndMAC()
    adv_enc_mac = EncryptAndMAC_Adv()
    
    # Vamos correr o jogo 1000 vezes. 
    # Como o adversário descobriu o problema do MAC sobre o texto limpo vai ganhar sempre
    calcular_vantagem(cifra_enc_mac, adv_enc_mac, iteracoes=1000)